# Fongbe ASR Training - Google Colab

Fine-tuning Whisper for Fongbe speech recognition using LoRA.

## Prerequisites
1. Runtime → GPU (T4 recommended)
2. Upload `fongbe_dataset.tar.gz` to `MyDrive/fongbe/`

## 1. Setup Environment

In [12]:
from pathlib import Path
from google.colab import drive
import subprocess
import tarfile

# Mount Drive
drive.mount('/content/drive')

# Config paths
PROJECT_ROOT = Path('/content/drive/MyDrive/fongbe')
DATASET_TAR = PROJECT_ROOT / 'fongbe_dataset.tar.gz'
DATA_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'fongbe_asr_unified'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
REPO_URL = 'https://github.com/Appolinairee/fongbe-asr.git'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"✓ Project: {PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Project: /content/drive/MyDrive/fongbe


## 2. Clone Repository

In [13]:
import os

os.chdir('/content')
if not Path('fongbe-asr').exists():
    !git clone {REPO_URL} fongbe-asr
else:
    !cd fongbe-asr && git pull

os.chdir('fongbe-asr')
print(f"✓ Working dir: {Path.cwd()}")

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 1), reused 3 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 3.73 KiB | 3.73 MiB/s, done.
From https://github.com/Appolinairee/fongbe-asr
   ac365ea..a5dc568  main       -> origin/main
Updating ac365ea..a5dc568
Fast-forward
 colab_training.ipynb | 203 +++++++++++++++++++++++++++++++++++++++++++++++----
 1 file changed, 189 insertions(+), 14 deletions(-)
✓ Working dir: /content/fongbe-asr


## 3. Extract Dataset

In [14]:
if not (DATA_ROOT / 'train').exists():
    if DATASET_TAR.exists():
        print(f"Extracting {DATASET_TAR.name}...")
        with tarfile.open(DATASET_TAR, 'r:gz') as tar:
            tar.extractall(PROJECT_ROOT / 'data' / 'processed')
        print("✓ Dataset extracted")
    else:
        raise FileNotFoundError(f"Dataset not found: {DATASET_TAR}")
else:
    print("✓ Dataset already extracted")

# Verify
n_train = len(list((DATA_ROOT / 'train').glob('*.arrow')))
print(f"✓ {n_train} files in train/")

✓ Dataset already extracted
✓ 1 files in train/


## 4. Install Dependencies

In [15]:
!pip install -q transformers accelerate peft evaluate jiwer datasets tensorboard soundfile librosa
print("✓ Dependencies installed")

✓ Dependencies installed


In [16]:
# Fix torchao version
!pip install -q --upgrade torchao
print("✓ torchao upgraded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 70.2 MB/s eta 0:00:00:00:01
✓ torchao upgraded


## 5. Verify GPU

In [17]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Runtime → Change runtime type → GPU")

CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB


## 6. Run Training

In [18]:
# Set paths for training script
import os
os.environ['DATASET_PATH'] = str(DATA_ROOT)
os.environ['OUTPUT_DIR'] = str(OUTPUT_ROOT / 'whisper-fongbe')

!python scripts/finetune_whisper.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
🔧 FINETUNING WHISPER FONGBE
Modèle: openai/whisper-small
LoRA rank: 8
Target modules: ['q_proj', 'v_proj']

📦 Chargement dataset...
✅ Dataset chargé:
   Train: 10864 samples
   Validation: 1358 samples
   Test: 1359 samples

🤖 Chargement Whisper...
Loading weights: 100% 479/479 [00:00<00:00, 5657.59it/s]
✅ Modèle chargé: openai/whisper-small
   Paramètres totaux: 241,734,912

🔬 Application LoRA...
✅ LoRA appliqué:
   Paramètres entraînables: 884,736 (0.36%)
   Paramètres gelés: 241,734,912

⚙️  Configuration preprocessing...
🔄 Preprocessing dataset...
Map (num_proc=4):   0% 0/10864 [

## 7. TensorBoard (Optional)

In [19]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_ROOT / 'whisper-fongbe'}

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 4502), started 0:03:40 ago. (Use '!kill 4502' to kill it.)

<IPython.core.display.Javascript object>